---
jupyter: python3
format:
  html:
    code-fold: true
    code-tools: true
bibliography: ../../references.bib
author:
  - name: Joshua Rhodes
    id: jr
    orcid: XXXX
    email: joshua.m.rhodes@durham.ac.uk
    corresponding: true
    degrees: PhD
    affiliation:
      - name: Durham University
        city: Durham
        country: United Kingdom
        url: www.durham.ac.uk
      - name: The Alan Turing Institute
        city: London
        country: United Kingdom
        url: www.turing.ac.uk

---

In [94]:
import pandas as pd
from urllib.request import urlretrieve
import zipfile

# How to use AddressGB

Download AddressGB and extract the files from the zip archive if you haven't already done so.

In [20]:
addressgb_archive_name = "addressgb.zip"
download_location = f"../data/{addressgb_archive_name}"
addressgb_extract_location = f"../data/addressgb"


urlretrieve(f"https://zenodo.org/api/records/10473597/files/{addressgb_archive_name}/content", download_location)

with zipfile.ZipFile(addressgb_archive_name,"r") as zip_ref:
    zip_ref.extractall("addressgb")

EW_1851_gb1900.tsv


('addressgb.zip', <http.client.HTTPMessage at 0x7fa799089ee0>)

In [2]:
censuses = {"EW":[1851,1861,1881,1891,1901,1911],
            "scot":[1851,1861,1871,1881,1891,1901]}

censuses = {"EW":[1851,],}
            # "scot":[1851,1861,1871,1881,1891,1901]}

In [63]:
def get_geom_uid(geom):
    if geom == "gb1900":
        geom_uid = "gb1900_uid"
    elif geom == "osopenroads":
        geom_uid = "openroads_uid"
    else:
        raise ValueError(f"Specified geometry should be either 'gb1900' or 'osopenroads' not 'geom'")
    
    return geom_uid

Download I-CeM from https://icem.ukdataservice.ac.uk/. Follow the instructions on the website. Also see https://www.campop.geog.cam.ac.uk/research/projects/icem/ for comprehensive documentation on I-CeM.

Set the parameters for reading the census file

In [127]:
icem_csv_params = {
		"sep": "\t",
		"encoding": "latin-1",
		"quoting": 3,
		"na_values": [".", " ", "-"],
		"usecols": ["recid","Age"], #add other columns from census here
		# "nrows":12000 #comment this line out to read all the icem data in your file
	}

Link AddressGB recid lookup files (e.g. `EW_1851_osopenroads_recidlkup.tsv`) to corresponding I-CeM file (e.g. `EW_1851`)

In [66]:
# Define function for linking addressgb recidlkup files to icem

def link_addressgb_to_icem(icem, addressgb, geom, lkup_type = "slim"):
    """
    addressgb:
        DataFrame of addressgb data
    icem:
        DataFrame of icem data
    lkup_type:
        Link all addressgb columns (full) or only targetgeom uid (slim). Default is slim."""

    if lkup_type == "slim":
        addressgb_linker = addressgb[["RecID", get_geom_uid(geom)]]
    else:
        addressgb_linker = addressgb

    icem_addressgb = pd.merge(left = icem,
                            right = addressgb_linker,
                            left_on = "recid",
                            right_on = "RecID",
                            how = "left",
                            validate = "one_to_one")
    
    icem_addressgb = icem_addressgb.drop(columns = ["RecID"])

    return icem_addressgb

Link this analysis/aggregated data to the corresponding geometry file in AddressGB

E.g. to `EW_1851_osopenroads.tsv` or `EW_1851_gb1900.tsv`

In [119]:
#Define a function that links the aggregated data to the geometry files in AddressGB
def link_addressgb_recid_geog(analysis_df, addressgb_geom, geom):

    addressgb_recid_geog = pd.merge(left = analysis_df,
                            right = addressgb_geom,
                            left_on = get_geom_uid(geom),
                            right_on = get_geom_uid(geom),
                            how = "left",
                            validate = "one_to_one")
    
    return addressgb_recid_geog

In [130]:
census_country = "EW"
census_year = 1851

icem = pd.read_csv(f"../../../../icem/2024/1851_ew_adj.txt",**icem_csv_params) #change path to where you census file is

for geom in ["gb1900", "osopenroads"]:
    file_name = f"addressgb/{census_country}_{census_year}_{geom}_recidlkup.tsv"
    addressgb_recidlkup = pd.read_csv(file_name, sep = "\t")
    testicem_addressgb = link_addressgb_to_icem(icem,
                                                addressgb_recidlkup,
                                                geom)
    
    """ Perform some analysis or aggregration on the street / address level data (e.g. on the `gb1900_uid` or `openroads_uid` columns) 
    """
    testicem_addressgb = testicem_addressgb[testicem_addressgb["Age"] <110].copy() #remove missing or spurious ages

    age_analysis = testicem_addressgb.groupby(by = [get_geom_uid(geom)])["Age"].mean().reset_index()
    age_analysis

    """ End of analysis section"""


    #Read AddressGB geom file

    geom_file_name = f"addressgb/{census_country}_{census_year}_{geom}.tsv"
    addressgb_geom = pd.read_csv(geom_file_name, sep = "\t", usecols = [get_geom_uid(geom), "geometry"])

    #Link AddressGB geom file to analysis dataframe
    geom_output = link_addressgb_recid_geog(age_analysis, addressgb_geom, geom)

    # Write linked output to file (N.B. Projection is in EPSG:27700, see https://epsg.io/27700)
    geom_output.to_csv(f"ageanalysis_{geom}.tsv", sep = "\t", index = False)